# Razonamiento no monótono: retractarse cuando llega un dato nuevo

**Facsímil 12 · Razonamiento simbólico** — capítulo 6
(monotonía frente a no monotonía, negación por fallo, lógica por defecto de Reiter, revisión de
creencias AGM y argumentación de Dung).

La lógica clásica tiene una virtud incómoda: nunca se arrepiente. Si de unas premisas se sigue una
conclusión, esa conclusión sobrevive a cualquier premisa que añadas después. A eso se le llama
**monotonía**, y es justo lo contrario de cómo razona una persona. «Los pájaros vuelan; Tweety es un
pájaro; luego Tweety vuela.» Te enteras de que Tweety es un pingüino y, sin dudarlo, **te retractas**:
no vuela. No es que te equivocaras antes; es que razonabas con lo que sabías, y al saber más cambiaste
de opinión. Esa capacidad de **deshacer conclusiones** es el corazón del razonamiento *no monótono*.

En este cuaderno la construyes desde cero, solo con Python. Verás por qué la lógica clásica no puede
retractarse, e implementarás cuatro maneras de que una máquina sí pueda: la **negación por fallo** (un
mini-Prolog que concluye «no vuela» porque *no consigue probar* que vuela), la **lógica por defecto**
de Reiter (reglas con excepciones que calculan una *extensión* y la rehacen al conocer el pingüino), la
**revisión de creencias** AGM (cómo añadir un hecho que contradice lo que creías sin que todo reviente)
y la **argumentación** de Dung (argumentos que se atacan, y cuáles sobreviven). Sin magia: estructuras
de datos, unas reglas y una búsqueda.

### Qué vas a aprender
- La diferencia exacta entre **monotonía** y **no monotonía**, comprobada con un razonador clásico que
  nunca quita conclusiones frente a uno por defecto que sí.
- La **negación por fallo** y la **suposición de mundo cerrado**: concluir lo negativo a partir de la
  incapacidad de probar lo positivo, con un mini-motor estilo Prolog.
- La **lógica por defecto** de Reiter: reglas por defecto con justificación, el cálculo de la
  **extensión** y la aparición de **varias extensiones** cuando los defaults chocan (el rombo de Nixon).
- La **revisión de creencias** AGM con un orden de **atrincheramiento**: expansión, contracción y
  revisión por la identidad de Levi, con la traza de qué creencia se sacrifica.
- La **argumentación abstracta** de Dung: conjuntos sin conflictos que se defienden solos, la extensión
  *fundamentada* y las *preferidas*, y cómo un argumento **reinstala** a otro.

### Cómo se usa este cuaderno
Las celdas vienen **sin ejecutar**: pulsa el botón de ejecutar (o `Mayús+Enter`) en cada una, de arriba
abajo. Lee el texto antes de correr el código; la gracia no es que «salga», sino entender *por qué* una
conclusión que parecía firme se cae cuando aparece un dato nuevo.

### Cuánto cuesta
Unos 30 minutos con calma. Python puro con un solo gráfico en Matplotlib; CPU, sin GPU ni cuentas.

> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

## 1. Monotonía: la lógica clásica no sabe retractarse

Empecemos por ver el problema. Un razonador clásico con reglas encadena hechos hasta agotar lo que
puede deducir: ese conjunto cerrado es su **clausura**. Su propiedad definitoria es la **monotonía**:
si añades premisas, el conjunto de conclusiones solo puede **crecer**, nunca encoger. Lo comprobamos
con la historia de Tweety, codificada con tres reglas: un ave vuela, un pingüino es un ave, y un
pingüino no vuela.

In [ ]:
def cierra(hechos, reglas):
    """Clausura hacia delante: dispara las reglas cuyas premisas ya estan en el conjunto."""
    s = set(hechos)
    cambio = True
    while cambio:
        cambio = False
        for premisas, consecuencia in reglas:
            if premisas <= s and consecuencia not in s:
                s.add(consecuencia)
                cambio = True
    return s

# Reglas clasicas (estrictas, sin excepciones): si X, entonces Y.
reglas_clasicas = [
    ({'ave'},      'vuela'),       # un ave vuela
    ({'pinguino'}, 'ave'),         # un pinguino es un ave
    ({'pinguino'}, 'no_vuela'),    # un pinguino no vuela
]

c1 = cierra({'ave'}, reglas_clasicas)               # solo se que es un ave
c2 = cierra({'ave', 'pinguino'}, reglas_clasicas)   # ademas, que es un pinguino

print("conclusiones desde {ave}          :", sorted(c1))
print("conclusiones desde {ave, pinguino}:", sorted(c2))
print()
print("¿c1 esta contenido en c2? (monotonía):", c1 <= c2)
print("¿'vuela' sigue ahí al añadir 'pinguino'?:", 'vuela' in c2)
print("¿aparece a la vez 'vuela' y 'no_vuela'?:", 'vuela' in c2 and 'no_vuela' in c2)

Ahí está el dilema. Al añadir «pingüino», la conclusión «vuela» **no se va**: la monotonía la protege.
Y como las reglas también obligan a «no vuela», el razonador acaba afirmando las dos cosas a la vez,
una **contradicción**. La lógica clásica solo tiene dos salidas, y ninguna sirve: o se calla la regla
«las aves vuelan» (y entonces no deduce nada útil de la mayoría de aves), o se contradice.

Lo que queremos es otra cosa: que «las aves vuelan» valga **por defecto**, salvo prueba en contra. Un
razonador así trata esa regla como una suposición revisable. Con solo eso, mira cómo cambia el final.

In [ ]:
def consistente_set(s, opuestos):
    """¿El conjunto s evita todo par de atomos contradictorios?"""
    return not any(a in s and b in s for a, b in opuestos)

def razonador_defecto(hechos, estrictas, defaults, opuestos):
    """Cierra los hechos con las reglas estrictas y luego aplica los defaults
    cuya conclusion sea consistente con lo ya sabido."""
    E = cierra(hechos, estrictas)
    cambio = True
    while cambio:
        cambio = False
        for premisas, conclusion in defaults:
            if premisas <= E and conclusion not in E and consistente_set(E | {conclusion}, opuestos):
                E = cierra(E | {conclusion}, estrictas)
                cambio = True
    return E

estrictas = [({'pinguino'}, 'ave'), ({'pinguino'}, 'no_vuela')]  # hechos seguros
defaults  = [({'ave'}, 'vuela')]                                  # las aves vuelan... por defecto
opuestos  = [('vuela', 'no_vuela')]                               # no pueden ser ciertas a la vez

E_ave     = razonador_defecto({'ave'},      estrictas, defaults, opuestos)
E_pinguino = razonador_defecto({'pinguino'}, estrictas, defaults, opuestos)

print("solo sé que es un ave   :", sorted(E_ave),      "->  ¿vuela?", 'vuela' in E_ave)
print("además es un pingüino   :", sorted(E_pinguino), "->  ¿vuela?", 'vuela' in E_pinguino)
print()
print("La conclusión 'vuela' estaba y desaparece al conocer 'pinguino': eso es NO monotonía.")

## 2. Negación por fallo y mundo cerrado

La primera forma práctica de razonar así nació en Prolog: la **negación por fallo**. La idea es
audaz: para concluir «**no** vuela(x)», no buscamos una prueba de que no vuela; nos basta con
**fracasar** al intentar probar «vuela(x)». Si tras agotar todas las reglas no conseguimos demostrarlo,
lo damos por falso. Detrás hay una **suposición de mundo cerrado**: lo que no consta y no se deduce, se
considera falso. Es como una base de datos: si tu nombre no está en la lista de invitados, no estás
invitado, y nadie tiene que escribir «fulano no está invitado».

Montamos un mini-motor estilo Prolog sobre hechos y reglas. En el cuerpo de una regla, un literal puede
ser positivo (`pos`, hay que probarlo) o negado por fallo (`naf`, se cumple si *no* se puede probar).

In [ ]:
def demuestra(meta, hechos, reglas, visitando=None):
    """¿Se puede probar 'meta'? 'naf' tiene exito justo cuando su atomo NO se prueba."""
    if visitando is None:
        visitando = set()
    if meta in hechos:
        return True
    if meta in visitando:           # corta posibles ciclos
        return False
    visitando = visitando | {meta}
    for cabeza, cuerpo in reglas:
        if cabeza != meta:
            continue
        ok = True
        for tipo, atomo in cuerpo:
            if tipo == 'pos':
                if not demuestra(atomo, hechos, reglas, visitando):
                    ok = False
                    break
            else:  # 'naf': negacion por fallo
                if demuestra(atomo, hechos, reglas, visitando):
                    ok = False
                    break
        if ok:
            return True
    return False

# Base de conocimiento (atomos ya instanciados por individuo).
hechos = {'ave_piolin', 'ave_pingu', 'pinguino_pingu'}
reglas = [
    ('anormal_pingu',  [('pos', 'pinguino_pingu')]),    # un pinguino es un ave "anormal"
    ('anormal_piolin', [('pos', 'pinguino_piolin')]),   # piolin no consta como pinguino
    ('vuela_piolin',   [('pos', 'ave_piolin'), ('naf', 'anormal_piolin')]),
    ('vuela_pingu',    [('pos', 'ave_pingu'),  ('naf', 'anormal_pingu')]),
]

print("¿vuela piolin? ->", demuestra('vuela_piolin', hechos, reglas))
print("¿vuela pingu?  ->", demuestra('vuela_pingu',  hechos, reglas))
print()
print("Mundo cerrado: 'no vuela(pingu)' se concluye porque NO se puede probar 'vuela(pingu)'.")
print("   ¿se prueba 'vuela_pingu'?", demuestra('vuela_pingu', hechos, reglas),
      " ->  luego concluimos: no_vuela(pingu)")

Fíjate en piolín: vuela porque es un ave y **no logramos probar** que sea anormal (no consta que sea
pingüino). En cambio pingu no vuela: sí se prueba que es anormal, y eso bloquea la regla. La conclusión
negativa sale del **fallo de la prueba**, no de un hecho explícito.

Y es no monótono por construcción: añade un hecho nuevo y una conclusión positiva puede caerse. Lo
vemos quitando y poniendo el dato «pingu es un pingüino».

In [ ]:
hechos_sin = {'ave_pingu'}                      # solo sabemos que pingu es un ave
hechos_con = {'ave_pingu', 'pinguino_pingu'}    # ...y ahora, que es un pinguino

print("sin saber que es pingüino -> ¿vuela_pingu?:", demuestra('vuela_pingu', hechos_sin, reglas))
print("al saber que es pingüino  -> ¿vuela_pingu?:", demuestra('vuela_pingu', hechos_con, reglas))
print()
print("La misma consulta pasa de Verdadero a Falso al añadir un hecho: no monotonía pura.")

## 3. Lógica por defecto de Reiter: extensiones

La negación por fallo es práctica pero algo tosca. Raymond Reiter (1980) le dio forma rigurosa con la
**lógica por defecto**. Una **regla por defecto** se escribe

$$\frac{\alpha \;:\; \beta}{\gamma}$$

y se lee: «si se cumple el **prerrequisito** $\alpha$, y es **consistente** suponer la **justificación**
$\beta$, concluye $\gamma$». En las reglas *normales* la justificación y la conclusión coinciden
($\beta=\gamma$): «las aves vuelan, salvo que suponer que vuela sea inconsistente». El resultado de
aplicar todos los defaults posibles de forma consistente es una **extensión**: un conjunto de creencias
cerrado y coherente.

Calculamos las extensiones probando qué subconjuntos de defaults se pueden aplicar a la vez: cada
default incluido tiene su prerrequisito y su conclusión en la extensión, y cada default excluido está
**bloqueado** (su prerrequisito falla, o su conclusión sería inconsistente). Reusamos `cierra` y
`consistente_set` de antes.

In [ ]:
import itertools

def extensiones(hechos, estrictas, defaults, opuestos):
    """Todas las extensiones de Reiter para defaults normales (busqueda sobre subconjuntos)."""
    resultado = []
    n = len(defaults)
    for r in range(n + 1):
        for S in itertools.combinations(range(n), r):
            E = cierra(set(hechos) | {defaults[i][1] for i in S}, estrictas)
            if not consistente_set(E, opuestos):
                continue
            # los defaults elegidos deben ser aplicables y estar realmente dentro
            if not all(defaults[i][0] <= E and defaults[i][1] in E for i in S):
                continue
            # los no elegidos deben estar bloqueados (no se podrian añadir)
            bloqueados = True
            for i in range(n):
                if i in S:
                    continue
                pre, con = defaults[i]
                if pre <= E and consistente_set(E | {con}, opuestos):
                    bloqueados = False
                    break
            if bloqueados:
                E_fro = frozenset(E)
                if E_fro not in [frozenset(x) for x in resultado]:
                    resultado.append(E)
    return resultado

# Tweety, ahora con defaults de Reiter.
estrictas = [({'pinguino'}, 'ave'), ({'pinguino'}, 'no_vuela')]
defaults  = [({'ave'}, 'vuela')]          # default normal: ave : vuela / vuela
opuestos  = [('vuela', 'no_vuela')]

ext_ave = extensiones({'ave'}, estrictas, defaults, opuestos)
print("solo 'ave'      -> extensión(es):", [sorted(E) for E in ext_ave])
print("   ¿concluye 'vuela'?", any('vuela' in E for E in ext_ave))

Con solo «ave» la extensión es $\{\text{ave}, \text{vuela}\}$: el default se aplica porque suponer que
vuela es consistente. Al conocer que es un pingüino, las reglas estrictas fuerzan «no vuela», así que
suponer «vuela» dejaría de ser consistente: el default queda **bloqueado** y la extensión ya no
contiene «vuela». La misma teoría, un hecho más, y la conclusión se retracta sola.

In [ ]:
ext_pinguino = extensiones({'pinguino'}, estrictas, defaults, opuestos)
print("ahora 'pinguino' -> extensión(es):", [sorted(E) for E in ext_pinguino])
print("   ¿concluye 'vuela'?",    any('vuela' in E for E in ext_pinguino))
print("   ¿concluye 'no_vuela'?", any('no_vuela' in E for E in ext_pinguino))
print()
print("El default 'las aves vuelan' está bloqueado: suponer 'vuela' contradice 'no_vuela'.")

### Cuando los defaults chocan: varias extensiones

Lo interesante de Reiter es que a veces **no hay una sola respuesta razonable**. El ejemplo clásico es
el *rombo de Nixon*: Nixon era cuáquero (y los cuáqueros son pacifistas por defecto) y republicano (y
los republicanos no lo son por defecto). Las dos reglas son igual de buenas y se contradicen. La lógica
por defecto no inventa un desempate: produce **dos extensiones**, una en la que Nixon es pacifista y
otra en la que no. Es el sistema diciendo, con honestidad, «con esto no puedo decidir».

In [ ]:
hechos_nixon  = {'republicano', 'cuaquero'}
estrictas_n   = []          # sin reglas estrictas
defaults_n = [
    ({'republicano'}, 'no_pacifista'),   # los republicanos no son pacifistas (por defecto)
    ({'cuaquero'},    'pacifista'),      # los cuaqueros sí lo son (por defecto)
]
opuestos_n = [('pacifista', 'no_pacifista')]

ext_nixon = extensiones(hechos_nixon, estrictas_n, defaults_n, opuestos_n)
print("número de extensiones:", len(ext_nixon))
for i, E in enumerate(ext_nixon, 1):
    print(f"  extensión {i}: {sorted(E)}")
print()
print("¿pacifista en la 1ª?:", 'pacifista' in ext_nixon[0],
      " | ¿pacifista en la 2ª?:", 'pacifista' in ext_nixon[1])
print("Dos mundos coherentes e incompatibles: la teoría no elige por ti.")

## 4. Revisión de creencias (AGM)

Cambiemos de plano. Hasta ahora razonábamos *dentro* de una base fija. La teoría **AGM** (por Alchourrón,
Gärdenfors y Makinson, 1985) estudia algo distinto: **cómo cambiar la base** cuando llega información
nueva, sin que el conjunto se vuelva incoherente. Hay tres operaciones:

- **Expansión** ($K + \varphi$): añadir $\varphi$ sin más. Rápido, pero si $\varphi$ contradice lo que
  creías, la base queda inconsistente y, en lógica clásica, una base inconsistente lo «cree» todo.
- **Contracción** ($K - \varphi$): dejar de creer $\varphi$ retirando lo justo para que ya no se siga.
- **Revisión** ($K * \varphi$): incorporar $\varphi$ **manteniendo la consistencia**, aunque contradiga
  creencias previas. Es la operación estrella.

¿Qué se sacrifica cuando hay conflicto? Lo que menos te importe. AGM lo formaliza con un orden de
**atrincheramiento**: a cada creencia le das un rango, y al chocar caen primero las menos atrincheradas.
Para decidir qué se sigue de una base necesitamos un mini-motor proposicional; lo montamos con tablas de
verdad sobre las pocas variables del ejemplo.

In [ ]:
VARS = ['llueve', 'mojado']        # el universo proposicional del ejemplo

# Una formula es un par (nombre_legible, funcion que la evalua sobre una asignacion).
def atomo(nombre):   return (nombre, lambda a, n=nombre: a[n])
def noo(x):          return ('¬' + x[0], lambda a, f=x[1]: not f(a))
def imp(x, y):       return ('(' + x[0] + '→' + y[0] + ')',
                             lambda a, fx=x[1], fy=y[1]: (not fx(a)) or fy(a))

def modelos(formulas):
    ms = []
    for combo in itertools.product([False, True], repeat=len(VARS)):
        a = dict(zip(VARS, combo))
        if all(f[1](a) for f in formulas):
            ms.append(a)
    return ms

def consistente(formulas):
    return len(modelos(formulas)) > 0

def implica(formulas, phi):
    return all(phi[1](a) for a in modelos(formulas))   # cierto en todos los modelos

llueve = atomo('llueve')
mojado = atomo('mojado')

# Base de creencias K, con su rango de atrincheramiento (mayor = mas firme).
#   rango 1: "llueve" es una observacion puntual, facil de revisar
#   rango 2: "si llueve, la calle se moja" es una ley general, mas firme
K = [
    (llueve,             1),
    (imp(llueve, mojado), 2),
]

print("creencias en K:", [f[0] for f, _ in K])
print("¿K implica 'mojado'?:", implica([f for f, _ in K], mojado))
print("¿K es consistente?:",   consistente([f for f, _ in K]))

La base cree que llueve y que, si llueve, la calle se moja; por tanto **deduce** que la calle está
mojada. Ahora la prueba de fuego: ¿qué pasa si te asomas y ves la calle **seca**? Si te limitas a
**expandir** con «no mojado», la base afirma a la vez «mojado» y «no mojado»: revienta. La expansión
no protege la consistencia.

In [ ]:
def expande(base, formula, rango):
    return base + [(formula, rango)]

K_mas_seco = expande(K, noo(mojado), 3)       # añadimos "no mojado" sin mas
print("tras expandir con «¬mojado»:", [f[0] for f, _ in K_mas_seco])
print("¿sigue siendo consistente?:", consistente([f for f, _ in K_mas_seco]))
print("Una base inconsistente, en lógica clásica, lo implica TODO; por ejemplo 'llueve':",
      implica([f for f, _ in K_mas_seco], llueve),
      "y también '¬llueve':", implica([f for f, _ in K_mas_seco], noo(llueve)))

Para hacerlo bien hay que **contraer** primero. La contracción $K - \varphi$ retira creencias hasta que
$\varphi$ deje de seguirse, sacrificando las **menos atrincheradas**. La implementamos así: recorremos
las creencias de **mayor a menor** rango y vamos quedándonos con cada una mientras no vuelva a implicar
$\varphi$; las que reintroducirían $\varphi$ se descartan. Contraer «mojado» debería obligarnos a soltar
«llueve» (rango 1), conservando la ley general (rango 2).

In [ ]:
def contrae(base, phi):
    """K - phi: conserva primero lo mas atrincherado; descarta lo que reintroduzca phi."""
    ordenadas = sorted(base, key=lambda fr: -fr[1])     # de mayor a menor rango
    res, traza = [], []
    for f, r in ordenadas:
        if implica([g for g, _ in res] + [f], phi):
            traza.append(f"descarta  «{f[0]}» (rango {r}): reintroduciría «{phi[0]}»")
        else:
            res.append((f, r))
            traza.append(f"conserva  «{f[0]}» (rango {r})")
    return res, traza

K_menos_mojado, traza = contrae(K, mojado)
print("Contracción K − «mojado»:")
for linea in traza:
    print("   " + linea)
print()
print("creencias que quedan:", [f[0] for f, _ in K_menos_mojado])
print("¿queda implicado 'mojado'?:", implica([f for f, _ in K_menos_mojado], mojado),
      " | ¿sigue consistente?:", consistente([f for f, _ in K_menos_mojado]))

Con la contracción lista, la **revisión** sale casi gratis por la **identidad de Levi**:

$$K * \varphi \;=\; (K - \neg\varphi) + \varphi$$

«para incorporar $\varphi$ de forma consistente, primero deja de creer su negación, luego añádelo». Lo
aplicamos a la observación «la calle no está mojada» ($\varphi = \neg\text{mojado}$): contraemos
$\neg\varphi = \text{mojado}$ y luego expandimos con $\neg\text{mojado}$. El resultado tiene que ser
consistente y, además, llevarnos por *modus tollens* a retractar «llueve».

In [ ]:
def revisa(base, phi, contraer, rango):
    """K * phi por la identidad de Levi: contraer la negacion de phi, luego expandir con phi.
    'contraer' es la formula a contraer (la negacion de phi, ya escrita de forma legible)."""
    contraida, traza = contrae(base, contraer)
    return expande(contraida, phi, rango), traza

phi = noo(mojado)                                  # nueva evidencia: la calle NO esta mojada
K_rev, traza = revisa(K, phi, contraer=mojado, rango=3)

print("Revisión K * «¬mojado» (identidad de Levi):")
for linea in traza:
    print("   " + linea)
print()
print("creencias tras revisar:", [f[0] for f, _ in K_rev])
print("¿consistente?:",                 consistente([f for f, _ in K_rev]))
print("¿implica '¬llueve'? (se retracta 'llueve'):", implica([f for f, _ in K_rev], noo(llueve)))
print("modelos que quedan:", modelos([f for f, _ in K_rev]))

## 5. Argumentación de Dung: quién gana la discusión

Las personas no solemos razonar calculando extensiones lógicas: **discutimos**. Phan Minh Dung (1995)
capturó eso con una idea minimalista y poderosa: olvida el contenido de los argumentos y quédate solo
con quién **ataca** a quién. Tienes un grafo dirigido, y la pregunta es qué conjuntos de argumentos son
**defendibles juntos**. Las definiciones clave:

- **Sin conflictos**: ningún argumento del conjunto ataca a otro del conjunto.
- **Defiende** a un argumento: para cada atacante suyo, alguien del conjunto contraataca.
- **Admisible**: sin conflictos y que defiende a todos sus miembros.
- **Fundamentada** (*grounded*): la postura más prudente, la que solo acepta lo que está obligado a
  aceptar. Se obtiene partiendo de los argumentos que nadie ataca y añadiendo los que ellos defienden.

El ejemplo: **A** «el acusado es inocente»; **B** «un testigo lo vio» (ataca a A); **C** «ese testigo no
es fiable» (ataca a B). Intuitivamente, C tumba a B, y al caer B, A queda **reinstalado**.

In [ ]:
def ataca_a(S, b, ataques):
    return any((x, b) in ataques for x in S)

def sin_conflictos(S, ataques):
    return not any((x, y) in ataques for x in S for y in S)

def defiende(S, a, args, ataques):
    return all(ataca_a(S, b, ataques) for b in args if (b, a) in ataques)

def admisible(S, args, ataques):
    return sin_conflictos(S, ataques) and all(defiende(S, a, args, ataques) for a in S)

def caracteristica(S, args, ataques):
    """F(S): todos los argumentos que S logra defender."""
    return set(a for a in args if defiende(S, a, args, ataques))

def fundamentada(args, ataques):
    """Extension fundamentada: menor punto fijo de F, partiendo del conjunto vacio."""
    S = set()
    while True:
        nuevo = caracteristica(S, args, ataques)
        if nuevo == S:
            return S
        S = nuevo

args    = ['A', 'B', 'C']
ataques = {('B', 'A'), ('C', 'B')}        # B ataca a A; C ataca a B

g = fundamentada(args, ataques)
print("extensión fundamentada:", sorted(g))
print("   ¿acepta A (inocente)?:", 'A' in g)
print("   ¿acepta B (lo vio)?:",   'B' in g)
print("   ¿acepta C (no fiable)?:", 'C' in g)
print()
print("C no tiene atacantes -> entra; tumba a B -> B fuera; sin B, A queda defendido -> reinstalado.")

La extensión fundamentada acepta {A, C} y rechaza B: un caso de **reinstalación**, A vuelve al juego
porque su único atacante ha sido derrotado. Cuando el grafo no tiene ciclos, esta extensión decide sola
y sin ambigüedad.

Pero, igual que en Nixon, puede haber empates. Las extensiones **preferidas** son los conjuntos
admisibles *maximales*: las posturas defendibles más comprometidas que puedas sostener. Cuando dos
argumentos se atacan mutuamente, hay dos preferidas (una por cada bando) y la fundamentada se queda
vacía: no se atreve a elegir.

In [ ]:
def subconjuntos(args):
    for r in range(len(args) + 1):
        for combo in itertools.combinations(args, r):
            yield set(combo)

def preferidas(args, ataques):
    adm = [S for S in subconjuntos(args) if admisible(S, args, ataques)]
    return [S for S in adm if not any(S < T for T in adm)]   # admisibles maximales

print("Caso del juicio (A, B, C):")
print("   preferidas:  ", [sorted(S) for S in preferidas(args, ataques)])
print("   fundamentada:", sorted(fundamentada(args, ataques)))
print()

# Conflicto simétrico: x e y se atacan mutuamente.
args2    = ['x', 'y']
ataques2 = {('x', 'y'), ('y', 'x')}
print("Conflicto simétrico (x <-> y):")
print("   preferidas:  ", [sorted(S) for S in preferidas(args2, ataques2)])
print("   fundamentada:", sorted(fundamentada(args2, ataques2)), "(vacía: no decide)")

Por último, **dibujamos** el grafo del juicio. Ver los ataques como flechas deja claro el mecanismo de
la reinstalación: C apunta a B y B apunta a A, y el efecto en cadena decide quién queda en pie. Pintamos
en verde lo aceptado por la extensión fundamentada y en rojo lo rechazado.

In [ ]:
import matplotlib.pyplot as plt

pos = {'A': (0, 1), 'B': (2.2, 1), 'C': (4.4, 1)}
etiqueta = {'A': 'A: inocente', 'B': 'B: lo vio\nun testigo', 'C': 'C: testigo\nno fiable'}
aceptados = fundamentada(args, ataques)

fig, ax = plt.subplots(figsize=(9, 3.4))
for n, (x, y) in pos.items():
    color = '#bfe3bf' if n in aceptados else '#f0bdbd'
    ax.add_patch(plt.Circle((x, y), 0.45, fc=color, ec='#333333', lw=1.5, zorder=2))
    ax.text(x, y, n, ha='center', va='center', fontsize=15, fontweight='bold', zorder=3)
    ax.text(x, y - 0.78, etiqueta[n], ha='center', va='top', fontsize=9)

for (s, t) in ataques:
    (x1, y1), (x2, y2) = pos[s], pos[t]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='-|>', color='#b03030', lw=2.2,
                                shrinkA=26, shrinkB=26), zorder=1)
    ax.text((x1 + x2) / 2, y1 + 0.18, 'ataca', ha='center', fontsize=8, color='#b03030')

ax.set_xlim(-1, 5.4)
ax.set_ylim(-0.4, 2)
ax.axis('off')
ax.set_title('Grafo de ataques: verde = aceptado, rojo = rechazado\n'
             'C tumba a B y, sin B, A queda reinstalado (extensión fundamentada {A, C})')
plt.tight_layout()
plt.show()
print("aceptados (fundamentada):", sorted(aceptados), " | rechazado:",
      sorted(set(args) - aceptados))

## 6. Pruébalo tú

El andamiaje está montado; ahora rómpelo y obsérvalo reaccionar:

- **Otra excepción.** En el razonador por defecto, añade un default «los pingüinos nadan» y un hecho que
  lo haga aplicable; comprueba que conviven «no vuela» y «nada» en la misma extensión.
- **Un tercer default en Nixon.** Añade «los académicos son escépticos» y haz a Nixon también académico.
  ¿Cuántas extensiones salen ahora? ¿Cambian las que ya había?
- **Cambia el atrincheramiento.** En AGM, dale a «llueve» rango 3 y a la ley rango 1, y vuelve a revisar
  por «¬mojado». Ahora se sacrificará la ley general en vez de la observación: la traza te lo dirá.
- **Rompe la reinstalación.** En el grafo de Dung, añade un argumento D que ataque a C. ¿Sigue A
  reinstalado? Calcula de nuevo la extensión fundamentada y míralo.
- **Un ciclo de tres.** Define ataques x→y, y→z, z→x y mira qué dicen la fundamentada y las preferidas.
  Los ciclos impares se comportan muy distinto a los pares.

## 7. Errores comunes

- **Esperar que la lógica clásica se retracte.** No puede: es monótona por definición. Si una regla
  tiene excepciones, no la metas como regla estricta; modélala como un **default**.
- **Confundir «no probado» con «falso».** La negación por fallo concluye lo negativo desde la
  incapacidad de probar lo positivo. Es útil, pero descansa en la **suposición de mundo cerrado**: si tu
  base está incompleta, concluirás falsedades con total aplomo.
- **Creer que siempre hay una sola extensión.** Cuando los defaults chocan (Nixon), hay **varias**.
  Quedarte con una al azar y presentarla como «la» conclusión es esconder que el sistema no decidía.
- **Tratar la revisión como una expansión.** Añadir sin más un hecho que contradice la base la vuelve
  inconsistente, y entonces «se sigue» cualquier cosa. Revisar es **contraer y luego añadir** (Levi).
- **Olvidar el atrincheramiento.** Sin un orden de qué creencias son más firmes, la contracción no sabe
  qué soltar y cualquier elección parece igual de válida. El rango es quien decide el sacrificio.
- **Leer el grafo de Dung al revés.** Una flecha es un **ataque**, no una implicación. Que B ataque a A
  no dice nada de B hasta saber si alguien ataca a B: la aceptación se calcula, no se lee de un vistazo.

## 8. Qué te llevas

- La lógica clásica es **monótona**: añadir premisas nunca quita conclusiones. Lo viste fallar con
  Tweety, que acababa volando y no volando a la vez. El razonamiento humano es **no monótono**: se
  retracta, y eso es una virtud, no un defecto.
- La **negación por fallo** concluye lo negativo cuando fracasa la prueba de lo positivo, bajo la
  **suposición de mundo cerrado**. La **lógica por defecto** de Reiter lo formaliza con reglas que
  tienen justificación y calcula **extensiones**; cuando los defaults chocan, hay **más de una**.
- La revisión **AGM** separa expansión, contracción y revisión, y usa el **atrincheramiento** para
  decidir qué creencia se sacrifica al llegar un hecho que contradice la base. La identidad de Levi
  reduce la revisión a «contraer y luego añadir».
- La **argumentación de Dung** abstrae el debate a un grafo de ataques y define qué posturas son
  defendibles. La extensión **fundamentada** es la prudente; las **preferidas**, las comprometidas. La
  **reinstalación** explica por qué un argumento revive cuando cae su atacante.
- Todo esto no compite con la IA que aprende de datos: la complementa. Un sistema que decide con reglas
  revisables, que sabe **deshacer** una conclusión y justificar por qué, es justo lo que le falta a un
  modelo que solo predice. Razonar bien incluye saber cambiar de opinión.

**Para seguir:** este capítulo cierra el bloque de razonamiento simbólico del facsímil. Cada vez que un
sistema diga «creía X, pero con este dato nuevo ahora creo Y», recuerda que por dentro late alguno de
los mecanismos que acabas de programar.

---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*